# Multi-Head Attention

## Learning Objectives
- Understand the rationale and implementation of multi-head attention.
- Learn how multiple attention heads provide diverse feature extraction.

## Introduction
Multi-head attention splits queries, keys, and values into multiple smaller sets processed in parallel, capturing different aspects of relationships in sequences.

## Core Concepts

- Linear Projections: Instead of using just one set of Query, Key, and Value vectors for the entire model, Multi-Head Attention first creates several distinct sets. It does this by taking the input embeddings and passing them through different, learned linear projection layers for each "head." This projects the original information into different representation subspaces, allowing each head to specialize.

- Multiple Heads: Each of these projected sets of Q, K, and V vectors is then fed into its own attention mechanism, completely in parallel. This means that if you have 8 heads, you are running 8 independent attention calculations at the same time. Each head can learn to focus on different patterns in the data—for example, one head might capture long-distance dependencies, while another focuses on local, syntactic relationships.

- Concatenation: Once each head has produced its output (a weighted sum of its Value vectors), all of these output vectors are concatenated together into a single, larger vector. This vector now contains the combined knowledge from all the different attention heads. This is then typically passed through a final linear layer to produce the output of the multi-head attention block.

## Example
Implement simplified multi-head attention.

In [1]:
import tensorflow as tf

class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.depth = d_model // num_heads
        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)
        self.dense = tf.keras.layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q):
        batch_size = tf.shape(q)[0]
        Q = self.wq(q)
        K = self.wk(k)
        V = self.wv(v)
        Q = self.split_heads(Q, batch_size)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size)
        
        matmul_qk = tf.matmul(Q, K, transpose_b=True)
        dk = tf.cast(tf.shape(K)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, V)
        output = tf.transpose(output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(output, (batch_size, -1, self.num_heads * self.depth))
        final_output = self.dense(concat_attention)
        return final_output

# Create example
mha = MultiHeadAttention(d_model=64, num_heads=8)
dummy_q = tf.random.uniform((1, 10, 64))
dummy_k = tf.random.uniform((1, 10, 64))
dummy_v = tf.random.uniform((1, 10, 64))
output = mha(dummy_v, dummy_k, dummy_q)
print(f"Multi-Head Attention output shape: {output.shape}")

2025-08-28 19:09:12.241596: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2025-08-28 19:09:12.241617: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-08-28 19:09:12.241621: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
I0000 00:00:1756404552.241634 1020012 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1756404552.241656 1020012 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Multi-Head Attention output shape: (1, 10, 64)


## Exercise
Increase number of heads and observe how output shape changes.

In [ ]:
# Your code here

## Summary
- Multi-head attention enables models to attend to information from multiple representation subspaces.
- This enhances the expressive power of Transformers.


## Further Reading
- [Multi-Head Attention Explained](https://jalammar.github.io/illustrated-transformer/#multi-head-attention)
- [TensorFlow MultiHeadAttention API](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MultiHeadAttention)
